### Data Ingestion 
Before we can ask legal questions, the AI needs a database of legal knowledge

#### Goal
- understand how raw text is convert into a format AI can understand
- loading pdf files using langchain
- text splitting
- vectorization

In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyMuPDFLoader # used to load pdf documents into langchain format
import pymupdf # underlying library for the previous module
import pandas as pd
from IPython.display import display, HTML

load_dotenv()

PERSIST_DIRECTORY = "./chroma_db"

In [2]:
file_path = os.path.join(r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Indian_Constitution.pdf")

In [3]:
# splitting the document pagewise
loader = PyMuPDFLoader(file_path)
data = loader.load()
data[0]

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 0}, page_content='£ÉÉ®iÉ BÉEÉ ºÉÆÉÊ´ÉvÉÉxÉ \n[1\n, 2024\n]\nTHE CONSTITUTION OF INDIA\n[As on 1st May, 2024] \n2024\nGOVERNMENT OF INDIA\nMINISTRY OF LAW AND JUSTICE\nLEGISLATIVE DEPARTMENT, OFFICIAL LANGUAGES WING')

We index preamble seperatly

In [4]:
# articles short
#Pages 3 to 31 contains the  introduction
articles_short = data[3:31]
articles_short[0]

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 3}, page_content='THE CONSTITUTION OF INDIA\n____________                                                                    \nCONTENTS\n__________\n                                                                                          \nPREAMBLE\nPART I \nTHE UNION AND ITS TERRITORY\nARTICLES \n1.\nName and territory of the Union.\n  2. \nAdmission or establishment of new States. \n[2A.         Sikkim to be associated with the Union.—Omitted.]\n3.\nFormation of new States and alteration of areas, boundar

In [5]:
# preamble
preamble = data[31]
preamble

Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'total_pages': 402, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trapped': '', 'modDate': 'D:20240701112033Z', 'creationDate': '', 'page': 31}, page_content='THE CONSTITUTION OF INDIA \nPREAMBLE\nWE, THE PEOPLE OF INDIA, having solemnly resolved to constitute \nIndia into a \n1[SOVEREIGN SOCIALIST SECULAR DEMOCRATIC \nREPUBLIC] and to secure to all its citizens:\nJUSTICE, social, economic and political;\n \nLIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity;\nand to promote among them all\nFRATERNITY assuring the dignity of the individual and the 2[unity \nand integrity of the Nation];\nIN OUR CONS

In [6]:
articles = data[32:283]

In [7]:
print(articles[0].page_content)

2
PART I
THE UNION AND ITS TERRITORY
1. Name and territory of the Union.—(1) India, that is Bharat, 
shall be a Union of States.
1[(2) The States and the territories thereof shall be as specified in 
the First Schedule.]
(3) The territory of India shall comprise—
(a) the territories of the States; 
2[(b) the Union territories specified in the First Schedule; 
and]
(c) such other territories as may be acquired.
2. Admission or establishment of new States.—Parliament may 
by law admit into the Union, or establish, new States on such terms and 
conditions as it thinks fit.
3[2A. [Sikkim to be associated with the Union.] —Omitted by the 
Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]
3. Formation of new States and alteration of areas, 
boundaries or names of existing States.—Parliament may by law—
(a) form a new State by separation of territory from any 
State or by uniting two or more States or parts of States or by 
uniting any territory to a part of any State

In [8]:

##code to remove headers  and footers from pdf
import re
for art in articles:
  art.page_content = re.split(r'_{2,}', art.page_content)[0].strip()

we split the article based on parts since constitution is divided into 25 parts

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

separator = "PART "  # Common chapter heading pattern

# Initialize the RecursiveCharacterTextSplitter with your chapter separator
text_splitter = RecursiveCharacterTextSplitter(
    separators=[separator],  # List of separators used to split the document
    chunk_size=1000,         # Maximum size of each split chunk
    chunk_overlap=100          # No overlap between splits
)

# Concatenate the 'page_content' of all Document objects into a single string
text = "".join([doc.page_content for doc in articles])

# Split the combined text by chapters
chapters = text_splitter.split_text(text)
chapters = chapters[1:]

In [10]:
chapters[7]

'PART VII\n[The States in Part B of the First Schedule].112'

In [11]:
def split_article_wise(chapter):
  """ Split the text based on Articles"""
  sections = re.split(r'(?=\n\d+\[?\d*[a-zA-Z]?[a-zA-Z]?\.\s)', chapter)
  # Optional: Strip leading/trailing whitespaces from each section
  sections = [section.strip() for section in sections]
  return sections

In [12]:
articles=[]
for chapter in chapters:
  articles.append(split_article_wise(chapter))

In [13]:
def cleaning_articles(arti):
  """Cleaning the articles text for unwanted special characters"""
  cl_art=[]
  for i in arti:
    pattern = r'^\d+\['
    if re.match(pattern, i):
      pos=arti.index(i)
      i = i.replace(i[:2], "", 1)
      arti[pos] = i
      cl_art.append(i)
    else:
      cl_art.append(i)
  return cl_art

In [14]:
docs = [cleaning_articles(i) for i in articles]

In [15]:
def adding_tag(docs):
  '''For adding Article keyword before each article number'''
  d=[docs[0]]
  for i in range(1,len(docs)):
    docs[i] = "Article " + docs[i]
    d.append(docs[i])
  return d

In [16]:
documents = [adding_tag(i) for i in docs]

In [17]:
nn =[i[1:] for i in documents]

In [18]:
nn[0][0].split(".")[0]

'Article 1'

In [19]:
nn[0]

['Article 1. Name and territory of the Union.—(1) India, that is Bharat, \nshall be a Union of States.\n1[(2) The States and the territories thereof shall be as specified in \nthe First Schedule.]\n(3) The territory of India shall comprise—\n(a) the territories of the States; \n2[(b) the Union territories specified in the First Schedule; \nand]\n(c) such other territories as may be acquired.',
 'Article 2. Admission or establishment of new States.—Parliament may \nby law admit into the Union, or establish, new States on such terms and \nconditions as it thinks fit.',
 'Article 2A. [Sikkim to be associated with the Union.] —Omitted by the \nConstitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).]',
 'Article 3. Formation of new States and alteration of areas, \nboundaries or names of existing States.—Parliament may by law—\n(a) form a new State by separation of territory from any \nState or by uniting two or more States or parts of States or by \nuniting any territory 

In [20]:
[j.split(".")[0] for i in nn for j in i]

['Article 1',
 'Article 2',
 'Article 2A',
 'Article 3',
 'Article 4',
 'Article 5',
 'Article 6',
 'Article 7',
 'Article 8',
 'Article 9',
 'Article 10',
 'Article 11',
 'Article 12',
 'Article 13',
 'Article 14',
 'Article 15',
 'Article 16',
 'Article 17',
 'Article 18',
 'Article 19',
 'Article 20',
 'Article 21',
 'Article 21A',
 'Article 22',
 'Article 23',
 'Article 24',
 'Article 25',
 'Article 26',
 'Article 27',
 'Article 28',
 'Article 29',
 'Article 30',
 'Article 31',
 'Article 31A',
 'Article 31B',
 'Article 31C',
 'Article 32',
 'Article 132A',
 'Article 33',
 'Article 34',
 'Article 35',
 'Article 36',
 'Article 37',
 'Article 38',
 'Article 39',
 'Article 39A',
 'Article 40',
 'Article 41',
 'Article 42',
 'Article 43',
 'Article 43A',
 'Article 43B',
 'Article 44',
 'Article 45',
 'Article 46',
 'Article 47',
 'Article 48',
 'Article 48A',
 'Article 49',
 'Article 50',
 'Article 51',
 'Article 51A',
 'Article 52',
 'Article 53',
 'Article 54',
 'Article 55',
 'Articl

In [21]:
const = [" ".join(i) for i in documents]

In [22]:
const[0].replace("\n"," ")

'PART I THE UNION AND ITS TERRITORY Article 1. Name and territory of the Union.—(1) India, that is Bharat,  shall be a Union of States. 1[(2) The States and the territories thereof shall be as specified in  the First Schedule.] (3) The territory of India shall comprise— (a) the territories of the States;  2[(b) the Union territories specified in the First Schedule;  and] (c) such other territories as may be acquired. Article 2. Admission or establishment of new States.—Parliament may  by law admit into the Union, or establish, new States on such terms and  conditions as it thinks fit. Article 2A. [Sikkim to be associated with the Union.] —Omitted by the  Constitution (Thirty-sixth Amendment) Act, 1975, s. 5 (w.e.f. 26-4-1975).] Article 3. Formation of new States and alteration of areas,  boundaries or names of existing States.—Parliament may by law— (a) form a new State by separation of territory from any  State or by uniting two or more States or parts of States or by  uniting any ter

In [23]:
# code to generate chapters names in order to add in the metadata

z2_list = []
for chapter in chapters:
    z2 = chapter.split('\n')[0]
    z2_list.append(z2)

z2_list

['PART I',
 'PART II',
 'PART III ',
 'PART IV',
 'PART  IVA',
 'PART  V',
 'PART VI',
 'PART VII',
 'PART VIII ',
 'PART IX',
 'PART IXA',
 'PART IXB ',
 'PART X',
 'PART XI',
 'PART XII',
 'PART XIII',
 'PART XIV',
 'PART  XIVA',
 'PART XV',
 'PART  XVI',
 'PART XVII',
 'PART XVIII',
 'PART XIX',
 'PART XX',
 'PART XXI',
 'PART XXII']

In [24]:
len(z2_list)

26

In [25]:
part_names = ['The union and its territories',
 'Citizenship',
 'Fundamental rights',
 'Directive principles of state policy',
 'Fundamental duties ',
 'The union',
 'The states',
 'The States in Part B of the First Schedule',
 'The union territories',
 'The panchayats',
 'The municipalities',
 'The co-operative societies',
 'The scheduled and tribal areas',
 'Relations between the union and the states',
 'Finance, property, contracts and suits',
 'Trade, commerce and intercourse within the territory of india',
 'Services under the union and the states',
 'Tribunals',
 'Elections',
 'Special provisions relating to certain classes',
 'Official language',
 'Emergency provisions',
 'Miscellaneous',
 'Amendment of the constitution',
 'Temporary, transitional and special provisions',
 'Short title, commencement,authoritative text in hindi and repeals']

In [26]:
len(part_names)

26

In [27]:
z1_list = [" ".join(j.capitalize() for j in i.split(" ")) for i in part_names]

In [28]:
z2_list = [i.strip() for i in z2_list]

In [29]:
z2_list,z1_list

(['PART I',
  'PART II',
  'PART III',
  'PART IV',
  'PART  IVA',
  'PART  V',
  'PART VI',
  'PART VII',
  'PART VIII',
  'PART IX',
  'PART IXA',
  'PART IXB',
  'PART X',
  'PART XI',
  'PART XII',
  'PART XIII',
  'PART XIV',
  'PART  XIVA',
  'PART XV',
  'PART  XVI',
  'PART XVII',
  'PART XVIII',
  'PART XIX',
  'PART XX',
  'PART XXI',
  'PART XXII'],
 ['The Union And Its Territories',
  'Citizenship',
  'Fundamental Rights',
  'Directive Principles Of State Policy',
  'Fundamental Duties ',
  'The Union',
  'The States',
  'The States In Part B Of The First Schedule',
  'The Union Territories',
  'The Panchayats',
  'The Municipalities',
  'The Co-operative Societies',
  'The Scheduled And Tribal Areas',
  'Relations Between The Union And The States',
  'Finance, Property, Contracts And Suits',
  'Trade, Commerce And Intercourse Within The Territory Of India',
  'Services Under The Union And The States',
  'Tribunals',
  'Elections',
  'Special Provisions Relating To Certain 

We use proper chunking of 1000 and overlap of 300

In [30]:
from langchain_text_splitters import CharacterTextSplitter
chunked_list = []
t_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    length_function=len,
    is_separator_regex=True,
)

In [31]:
for i,chapter in enumerate(const):
  # adding meta data here li is the list with name of chapters which we created above
  metadatas = [{"source": "https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf",
                "source_name": "Indian Constitution", "country":"India","db_owner":"LawGlance","part":z2_list[i],"part_name":z1_list[i]}]
  # Use the splitter to split each chapter into chunks
  chunks = t_splitter.create_documents([chapter],metadatas=metadatas)
  # Append the chunks to the chunked_list
  chunked_list.append(chunks)

In [32]:
chunked_list[1]

[Document(metadata={'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'source_name': 'Indian Constitution', 'country': 'India', 'db_owner': 'LawGlance', 'part': 'PART II', 'part_name': 'Citizenship'}, page_content='PART II\nCITIZENSHIP Article 5. Citizenship at the commencement of the Constitution.—At the \ncommencement of this Constitution, every person who has his domicile in the \nterritory of India and—\n(a) who was born in the territory of India; or \n(b) either of whose parents was born in the territory of India; or\n(c) who has been ordinarily resident in the territory of India for \nnot less than five years immediately preceding such commencement,  \nshall be a citizen of India. Article 6. Rights of citizenship of certain persons who have migrated to \nIndia from Pakistan.—Notwithstanding anything in article 5, a person who \nhas migrated to the territory of India from the territory now included in \nPakistan sha

In [33]:
def flatten_extend(matrix):
    flat_list = []
    for row in matrix:
        flat_list.extend(row)
    return flat_list

In [34]:
do = flatten_extend(chunked_list)

we use huggingface embeddings in our production

In [35]:
# setup the llm
llm = ChatGroq(model= "llama-3.3-70b-versatile",
               temperature=0.9,
               api_key= os.getenv("GROQ_API_KEY") # Optional if the env var is named exactly GROQ_API_KEY
               ) 

embeddings =  HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
# class method used to initialize the database for the first time
# by providing the raw data
"""
What it does: It takes your list of Document objects, runs them through the embedding model to turn text into numbers (vectors), and then saves them into the specified persist_directory.

When to use it: Use this during the Ingestion Phase. You run this once to "build" your knowledge base.

Analogy: This is like buying a bookshelf and physically putting books on it for the first time.
"""

vectorstore = Chroma.from_documents(
    documents=do,
    embedding= embeddings,
    persist_directory="chroma_db_legal_bot_part1"
)


# vector_store.add_documents(documents=do)

In [37]:
# loads the vector store

"""
This is the Class Constructor used to connect to an existing database that has already been saved to your disk.

What it does: It looks at the persist_directory on your computer. If it finds an existing Chroma database there, it links it to your code using the embedding_function so it knows how to "read" the stored vectors. It does not add new documents; it just prepares the database for searching.

When to use it: Use this in your Main Application (lawglance_main.py or app.py). You don't want to re-process the PDF every time a user asks a question; you just want to open the existing database.

Analogy: This is like walking up to a bookshelf that is already full and opening it so you can look for information.
"""

vector_store = Chroma(
    persist_directory="chroma_db_legal_bot_part1",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(
    search_type = "similarity", 
    search_kwargs={"k": 10}
)

retriever.invoke("What is article 1")

[Document(id='9d733ae9-8fa6-4f94-bfc7-6fd94c9962ed', metadata={'part_name': 'Emergency Provisions', 'country': 'India', 'source_name': 'Indian Constitution', 'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'db_owner': 'LawGlance', 'part': 'PART XVIII'}, page_content='containing such a recital.] Article 359. Suspension of the enforcement of the rights conferred by \nPart III during emergencies.—(1) Where a Proclamation of Emergency is in \noperation, the President may by order declare that the right to move any court \nfor the enforcement of such of 1[the rights conferred by Part III (except articles \n20 and 21)] as may be mentioned in the order and all proceedings pending in \nany court for the enforcement of the rights so mentioned shall remain \nsuspended for the period during which the Proclamation is in force or for such \nshorter period as may be specified in the order.\n2[(1A) While an order made under clause (1

In [38]:
def footer_remover(dat):
  for art in dat:
    art.page_content = re.split(r'_{2,}', art.page_content)[0].strip()
  return(" ".join([i.page_content for i in dat]))

In [39]:
first_schedule = footer_remover(data[283:294])

In [40]:
second_schedule = footer_remover(data[293:300])
print(second_schedule)

263
SECOND SCHEDULE
[Articles 59(3), 65(3), 75(6), 97, 125, 148(3), 158(3), 164 (5), 186 and 221] 
PART A 
PROVISIONS AS TO THE PRESIDENT AND THE GOVERNORS OF STATES 1***
1. There shall be paid to the President and to the Governors of the States
1***  the following emoluments per mensem, that is to say:—
The President
……                10,000 rupees .
The Governor of a State     ……   
5,500 rupees .
2. There shall also be paid to the President and to the Governors of the 
States 2***  such allowances as were payable respectively to the Governor-
General of the Dominion of India and to the Governors of the corresponding 
Provinces immediately before the commencement of this Constitution.
3. The President and the Governors of 3[the States] throughout their respective 
terms of office shall be entitled to the same privileges to which the Governor-
General and the Governors of the corresponding Provinces were respectively 
entitled immediately before the commencement of this Constitution.


In [41]:
third_schedule = footer_remover(data[299:303])
third_schedule

'269\nTHIRD SCHEDULE\n[Articles 75(4), 99, 124(6), 148(2), 164(3), 188 and 219]   \nForms  of  Oaths  or Affirmations \nI\nForm of oath of office for a Minister for the Union:—\n“I,   A. B.,  do  swear   in   the  name   of   God  that I will bear true faith\n                                  solemnly affirm   \nand allegiance to  the  Constitution of India as by law established, 1[that I \nwill uphold the sovereignty and integrity of India,] that I will faithfully \nand conscientiously discharge my duties as a Minister for the Union and \nthat I will do right to all manner of people in accordance with the \nConstitution and the law, without fear or favour, affection or ill-will.”\nII\nForm of oath of secrecy for a Minister for the Union:—\n“I,   A.B.,  do swear  in  the  name  of  God that  I  will  not  directly  or\nsolemnly affirm\nindirectly communicate or reveal to any person or persons any matter \nwhich shall be brought under my consideration or shall become known \nto me as a 

In [42]:
fourth_schedule = footer_remover(data[303:306])
print(fourth_schedule)

273
1[FOURTH SCHEDULE
[Articles 4(1) and 80(2)] 
Allocation of seats in the Council of States
To each State or Union territory specified in the first column of  the 
following table, there shall be allotted the number of seats specified in the 
second column thereof opposite to that State or that Union territory, as the case 
may be: 
TABLE
 1.
Andhra Pradesh ...................................................
2[11]
3[2.
Telangana ...........................................................
7]
4[3.]
Assam .................................................................
7
4[4.]
Bihar ...................................................................
5[16]
6[4[5.]
Jharkhand ............................................................
6]
7[8[4[6.]
Goa .....................................................................
1]]
9[8[4[7.]
Gujarat ................................................................
11]]
10[8[4[8.]
Haryana ...........................................................

In [43]:
fifth_schedule = footer_remover(data[306:310])
sixth_schedule = footer_remover(data[310:340])
seventh_schedule = footer_remover(data[340:355])
eighth_schedule = footer_remover(data[355:357])
ninth_schedule = footer_remover(data[357:375])
tenth_schedule = footer_remover(data[375:379])
eleventh_schedule =footer_remover(data[379:380])
twelfth_schedule = footer_remover(data[380:381])

In [44]:
schedules = []
schedules.append(first_schedule)
schedules.append(second_schedule)
schedules.append(third_schedule)
schedules.append(fourth_schedule)
schedules.append(fifth_schedule)
schedules.append(sixth_schedule)
schedules.append(seventh_schedule)
schedules.append(eighth_schedule)
schedules.append(ninth_schedule)
schedules.append(tenth_schedule)
schedules.append(eleventh_schedule)
schedules.append(twelfth_schedule)

In [45]:
print(schedules[0])

253
 
1[FIRST SCHEDULE 
[Articles 1 and 4]
I. THE STATES 
 
Name
Territories
1. 
Andhra 
Pradesh
 
 
2[The territories specified in sub-section (1) of section 3 of 
the Andhra State Act, 1953, sub-section (1) of section 3 of  
the States Reorganisation Act, 1956, the First Schedule to 
the Andhra Pradesh and Madras (Alteration of Boundaries) 
Act, 1959, and the Schedule to the Andhra Pradesh and 
Mysore (Transfer of Territory) Act, 1968, but excluding 
the territories specified in the Second Schedule to the 
Andhra Pradesh and Madras (Alteration of Boundaries) 
Act, 1959] 3[and the territories specified in section 3 of 
the Andhra Pradesh Reorganisation Act, 2014].
2. Assam
The 
territories 
which 
immediately 
before 
the 
commencement of this Constitution were comprised in the 
Province of Assam, the Khasi States and the Assam Tribal 
Areas, 
but 
excluding 
the 
territories 
specified 
in 
the 
Schedule 
to 
the 
Assam 
(Alteration of Boundaries) Act, 1951 4[and the territories 
spe

In [46]:
schedule_names = [
    "schedule 1",
    "schedule 2",
    "schedule 3",
    "schedule 4",
    "schedule 5",
    "schedule 6",
    "schedule 7",
    "schedule 8",
    "schedule 9",
    "schedule 10",
    "schedule 11",
    "schedule 12"
]

In [47]:
from langchain_text_splitters import CharacterTextSplitter
chucked_list = []
t_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=300,
    length_function=len,
    is_separator_regex=True,
)

for i,chapter in enumerate(schedules):
  # adding meta data here li is the list with name of chapters which we created above
  metadatas = [{"source": "https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf",
                "source_name": "Indian Constitution", "country":"India","db_owner":"LawGlance","schedule":schedule_names[i],}]
  # Use the splitter to split each chapter into chunks
  chunks = t_splitter.create_documents([chapter],metadatas=metadatas)
  # Append the chunks to the chunked_list
  chunked_list.append(chunks)


In [48]:
chunked_list[0]

[Document(metadata={'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'source_name': 'Indian Constitution', 'country': 'India', 'db_owner': 'LawGlance', 'part': 'PART I', 'part_name': 'The Union And Its Territories'}, page_content='PART I\nTHE UNION AND ITS TERRITORY Article 1. Name and territory of the Union.—(1) India, that is Bharat, \nshall be a Union of States.\n1[(2) The States and the territories thereof shall be as specified in \nthe First Schedule.]\n(3) The territory of India shall comprise—\n(a) the territories of the States; \n2[(b) the Union territories specified in the First Schedule; \nand]\n(c) such other territories as may be acquired. Article 2. Admission or establishment of new States.—Parliament may \nby law admit into the Union, or establish, new States on such terms and \nconditions as it thinks fit. Article 2A. [Sikkim to be associated with the Union.] —Omitted by the \nConstitution (Thirty-sixth A

In [49]:
def flatten_extend(matrix):
  flat_list = []
  for row in matrix:
    flat_list.extend(row)
  return flat_list
do = flatten_extend(chunked_list)
persistant_directory = "chroma_db_legal_bot_part1"
db1 = Chroma(persist_directory=persistant_directory, embedding_function=embeddings)
db1.add_documents(documents=do)

['62551750-728b-4b24-9ffc-7a7ac78e6993',
 '3d9c20d8-55b6-4a51-a4de-e052bc9d96ac',
 'c2bff44b-05ee-48ea-b488-c499e9edb56c',
 '2ecd4d7b-fd7e-475c-8949-6c295ae0ed4f',
 '71fcc37e-b0f7-433d-9da0-57326127c582',
 'cb52d55e-2b8a-47cd-bb60-9e68d2d9c7d0',
 '68234323-e628-4355-bc09-6121490c7998',
 'd88f5948-cbe4-4fb9-bcf0-d969508c91d7',
 '6586c559-7cd1-418c-a230-c2119a85bf9e',
 '239557a8-0f9b-429e-9145-006ab3c46ce6',
 '1cbfc56a-0143-41d9-8029-4e439ba4e06e',
 'ea109620-d83b-4d3c-8a49-792333cebf25',
 '5aeff88a-488f-4396-b0c8-2ab3ac874926',
 '70a5bc7c-bf34-4728-b65d-5b391336104c',
 '74c44c82-88d4-4dc7-9365-6845ce8dcd8f',
 '59ba3358-4e4a-447c-9153-58dd0a971628',
 '04865f92-806d-4ed2-ba25-7eb980ea8c88',
 '09e3ce57-b20b-4abe-8832-0807642a9dc1',
 '6fc8aff2-30e3-4cdb-8a88-c64e927bca77',
 '9dd31bdf-fc13-4c34-8a8e-25c853008d10',
 '72497ef4-a3ed-4892-97cd-06346351f252',
 '8d05adc4-2983-496a-aac8-b942d43fa8c3',
 '22210618-6256-4175-b5f2-651350a22ff2',
 '052228d3-b762-48c2-90b2-37515c85848d',
 'b15b37fe-1a44-

In [50]:
pr = data[3]
pr.metadata.clear()
pr

Document(metadata={}, page_content='THE CONSTITUTION OF INDIA\n____________                                                                    \nCONTENTS\n__________\n                                                                                          \nPREAMBLE\nPART I \nTHE UNION AND ITS TERRITORY\nARTICLES \n1.\nName and territory of the Union.\n  2. \nAdmission or establishment of new States. \n[2A.         Sikkim to be associated with the Union.—Omitted.]\n3.\nFormation of new States and alteration of areas, boundaries or \nnames of existing States.\n4.\nLaws made under articles 2 and 3 to provide for the amendment of \nthe First and the Fourth Schedules and supplemental, incidental \nand consequential matters.\nPART II\nCITIZENSHIP\n5.\nCitizenship at the commencement of the Constitution.\n6.\nRights of citizenship of certain persons who have migrated to \nIndia from Pakistan.\n7.\nRights of citizenship of certain migrants to Pakistan.\n8.\nRights of citizenship of certain p

In [51]:
# Delete all old metadata
# pr.metadata.clear()

# Add new metadata
pr.metadata = {"source": "https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf",
                "source_name": "Indian Constitution", "country":"India","db_owner":"LawGlance","preamble": "preamble",}

# Now the old metadata is completely removed and replaced with the new one


In [52]:
pr

Document(metadata={'source': 'https://cdnbbsr.s3waas.gov.in/s380537a945c7aaa788ccfcdf1b99b5d8f/uploads/2024/07/20240716890312078.pdf', 'source_name': 'Indian Constitution', 'country': 'India', 'db_owner': 'LawGlance', 'preamble': 'preamble'}, page_content='THE CONSTITUTION OF INDIA\n____________                                                                    \nCONTENTS\n__________\n                                                                                          \nPREAMBLE\nPART I \nTHE UNION AND ITS TERRITORY\nARTICLES \n1.\nName and territory of the Union.\n  2. \nAdmission or establishment of new States. \n[2A.         Sikkim to be associated with the Union.—Omitted.]\n3.\nFormation of new States and alteration of areas, boundaries or \nnames of existing States.\n4.\nLaws made under articles 2 and 3 to provide for the amendment of \nthe First and the Fourth Schedules and supplemental, incidental \nand consequential matters.\nPART II\nCITIZENSHIP\n5.\nCitizenship at the co

In [53]:
db1.add_documents(documents=[pr])

['eec56039-f892-452f-bfc6-f1b01b7aac44']

In [54]:
from langchain_core.documents import Document

In [55]:
greetings = [
    {"text": "Hello", "language": "English"},
    {"text": "Hi", "language": "English"},
    {"text": "Hey", "language": "English"},
    {"text": "Greetings", "language": "English"},
    {"text": "Good Morning", "language": "English"},
    {"text": "Good Afternoon", "language": "English"},
    {"text": "Good Evening", "language": "English"},
    {"text": "Howdy", "language": "English"},
    {"text": "What's up?", "language": "English"},
    {"text": "Welcome", "language": "English"},
    {"text": "Salutations", "language": "English"},
    {"text": "Hiya", "language": "English"},
    {"text": "Yo", "language": "English"},
    {"text": "How’s it going?", "language": "English"},
    {"text": "Nice to meet you", "language": "English"},
    {"text": "How are you?", "language": "English"},
    {"text": "Good day", "language": "English"},
    {"text": "Pleased to meet you", "language": "English"},
    {"text": "Hey there", "language": "English"},
    {"text": "What's new?", "language": "English"},
    {"text": "Long time no see", "language": "English"},
    {"text": "How have you been?", "language": "English"},
    {"text": "How do you do?", "language": "English"},
    {"text": "Good to see you", "language": "English"},
    {"text": "How are things?", "language": "English"},
    {"text": "Hello there", "language": "English"},
    {"text": "Hi there", "language": "English"},
]

# Convert the greetings into a list of LangChain Documents with language metadata
documents = [Document(page_content=greeting["text"], metadata={"source": "english greeting words","db_owner":"LawGlance","language": greeting["language"]})
    for greeting in greetings
]


### Using CrewAI in the legal domain

Thie notebook utilises an exemplary application of Crew AI in the legal domain, showcasing the power of multi-agent workflows to streamline complex legal research tasks. It combines Crew AI, LangChain, and Chroma to retrieve legal documents, perform web searches, and deliver concise, accurate answers tailored to user queries.

This notebook serves as a practical demonstration of how Crew AI can be applied in the legal field, making it a valuable resource for legal professionals, researchers, and developers exploring AI-driven solutions. Future enhancements include multilingual support, voice interaction, and mobile deployment, demonstrating its scalability and versatility.

### Agentic RAG Using CrewAI & LangChain for Legaldomain

In [56]:
# importing necessary libraries
from crewai import Agent, Task, Crew
from crewai.tools import tool 

In [57]:
vector_store = Chroma(
    persist_directory="chroma_db_legal_bot_part1",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(
    search_type = "similarity_score_threshold", 
    search_kwargs={
        "k": 10, 
        "score_threshold":0.3
        }
)

### We are creating a Custom tool for retrieval

In [58]:
import getpass


@tool("ChromaRetriever")
def chroma_retriever_tool(query : str):
    # Assuming 'retriever' is your Chroma retriever instance
    """Retrieves relevant documents from the Chroma vector store."""
    results = retriever.invoke(query)
    return results

chroma_tool = chroma_retriever_tool


#### Custom tool for web search
We are using Tavily web search

In [59]:
from langchain_community.tools import TavilySearchResults
from crewai.tools import BaseTool
from pydantic import Field

search = TavilySearchResults()

class SearchTool(BaseTool):
    name: str = "Search"
    description: str = "Useful for search-based queries. Use this to find current information about latest legal trends and news."
    search: TavilySearchResults = Field(default_factory=TavilySearchResults)

    def _run(self, query: str) -> str:
        """Execute the search query and return results"""
        try:
            return self.search.run(query)
        except Exception as e:
            return f"Error performing search: {str(e)}"
web_search_tool = SearchTool()

C:\Users\user\AppData\Local\Temp\ipykernel_15104\716671558.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults()


ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error